# Explore the EAF / GPU environment

**Kernel:** select `conda env:.conda-diffusion` (same as `AnalyzeDDIM2DDIM_Outputs.ipynb`).

This notebook runs on the remote EAF Jupyter kernel to map:
- hostname, CUDA, conda env
- `diffusion-anomaly` checkouts and training scripts
- `/scratch/7DayLifetime/munjung/...` data & inference products
- existing checkpoints and log dirs for learning curves

Re-run after connecting the kernel; paths discovered here feed `01_TrainDiffusion` / `02_TrainClassifier` / `01_RunInference`.


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import socket
import subprocess
from pathlib import Path

APP_ROOT = Path("/exp/sbnd/app/users/munjung/anomaly-detection")
print("APP_ROOT ", APP_ROOT)
print("hostname :", socket.gethostname())
print("user     :", os.environ.get("USER") or os.environ.get("USERNAME"))
print("cwd      :", Path.cwd())
print("python   :", shutil.which("python") or shutil.which("python3"))
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
try:
    import torch
    print("torch    :", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "| devices:", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("gpu[0]   :", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch import failed:", e)


## Locate diffusion-anomaly and flag scripts


In [ ]:
candidates = [
    Path.home() / "diffusion-anomaly",
    Path.home() / "anomaly-detection" / "train" / "diffusion-anomaly",
    Path("/exp/sbnd/app/users/munjung/anomaly-detection/train/diffusion-anomaly"),
    Path("/exp/sbnd/app/users/munjung/diffusion-anomaly"),
    Path("/scratch/7DayLifetime/munjung/diffusion-anomaly"),
    Path("/scratch/7DayLifetime/gputnam/diffusion-anomaly"),
]
# also search a few likely homes
extra_roots = [Path.home(), Path("/scratch/7DayLifetime/munjung"), Path("/exp/sbnd/app/users/munjung")]
for root in extra_roots:
    if not root.exists():
        continue
    for p in root.rglob("guided_diffusion"):
        if p.is_dir() and (p.parent / "scripts" / "image_train.py").exists():
            candidates.append(p.parent)

found = []
seen = set()
for c in candidates:
    c = c.resolve() if c.exists() else c
    key = str(c)
    if key in seen:
        continue
    seen.add(key)
    ok = c.exists() and (c / "guided_diffusion").is_dir()
    print(("OK " if ok else "--"), c)
    if ok:
        found.append(c)

DIFFUSION_ROOT = found[0] if found else None
print("\nSelected DIFFUSION_ROOT =", DIFFUSION_ROOT)
if DIFFUSION_ROOT:
    print("scripts:", sorted(p.name for p in (DIFFUSION_ROOT / "scripts").glob("*.py")))
    for sh in sorted(DIFFUSION_ROOT.glob("*flags*.sh")) + sorted(DIFFUSION_ROOT.glob("model_flags*.sh")):
        print("flags :", sh)


## Map scratch / data trees


In [ ]:
from pprint import pprint

roots = [
    Path("/scratch/7DayLifetime/munjung"),
    Path("/scratch/7DayLifetime/gputnam"),
    Path("/exp/sbnd/data/users/munjung/anomaly-detection"),
    Path("/exp/sbnd/data/users/gputnam/training-SBND"),
    Path("/exp/sbnd/app/users/munjung/anomaly-detection"),
]

for r in roots:
    print("\n====", r, "exists=", r.exists())
    if not r.exists():
        continue
    try:
        kids = sorted(r.iterdir(), key=lambda p: p.name)[:40]
        for k in kids:
            mark = "d" if k.is_dir() else "f"
            print(f"  [{mark}] {k.name}")
        if len(list(r.iterdir())) > 40:
            print("  ...")
    except PermissionError as e:
        print("  permission error:", e)


In [ ]:
# Deeper look at anomaly-detection scratch + ICARUS inference products
for r in [
    Path("/scratch/7DayLifetime/munjung/anomaly-detection"),
    Path("/scratch/7DayLifetime/munjung/ICARUS"),
]:
    print("\n====", r)
    if not r.exists():
        print("  missing")
        continue
    for p in sorted(r.glob("*"))[:60]:
        print(" ", "DIR " if p.is_dir() else "FILE", p.name)


## Training logs / checkpoints (for learning curves)


In [ ]:
import re

log_globs = [
    Path("/scratch/7DayLifetime/munjung").glob("**/log*.txt"),
    Path("/scratch/7DayLifetime/munjung").glob("**/progress.csv"),
    Path("/exp/sbnd/data/users/gputnam/training-SBND").glob("**/log*.txt"),
    Path("/exp/sbnd/data/users/gputnam/training-SBND").glob("**/*results*"),
]

# Cap search — scratch trees can be huge
hits = []
for root in [
    Path("/scratch/7DayLifetime/munjung/anomaly-detection"),
    Path("/exp/sbnd/data/users/gputnam/training-SBND"),
]:
    if not root.exists():
        continue
    for pat in ("**/progress.csv", "**/log.txt", "**/*.pt"):
        for i, p in enumerate(root.glob(pat)):
            hits.append(p)
            if i >= 30:
                break

print(f"sample hits ({len(hits)}):")
for p in hits[:40]:
    print(" ", p)


## Persist discovered paths for other notebooks


In [ ]:
import json
from pathlib import Path

# Probe whichever /scratch/7Day* pool actually exists on this node.
_pools = sorted(p for p in Path("/scratch").glob("7Day*") if p.is_dir()) if Path("/scratch").is_dir() else []
print("existing scratch pools:", _pools)
_scratch_anomaly = None
for _pool in _pools:
    cand = _pool / "munjung" / "anomaly-detection"
    if cand.is_dir():
        _scratch_anomaly = cand
        break
if _scratch_anomaly is None and _pools:
    # user dir present?
    for _pool in _pools:
        user = _pool / "munjung"
        if user.is_dir():
            _scratch_anomaly = user / "anomaly-detection"
            break

_scratch_icarus = None
for _pool in _pools:
    cand = _pool / "munjung" / "ICARUS"
    if cand.is_dir():
        _scratch_icarus = cand
        break

discover = {
    "hostname": socket.gethostname(),
    "diffusion_root": str(DIFFUSION_ROOT) if DIFFUSION_ROOT else None,
    "scratch_pools": [str(p) for p in _pools],
    "scratch_munjung": str(_scratch_anomaly.parent) if _scratch_anomaly else None,
    "scratch_anomaly": str(_scratch_anomaly) if _scratch_anomaly else None,
    "scratch_icarus": str(_scratch_icarus) if _scratch_icarus else None,
    "data_root": "/exp/sbnd/data/users/munjung/anomaly-detection",
    "app_root": str(APP_ROOT),
    "gputnam_training": "/exp/sbnd/data/users/gputnam/training-SBND",
    "train_flags_local": str(APP_ROOT / "configs" / "train_flags"),
    "train_flags_imported": str(APP_ROOT / "configs" / "train_flags" / "imported"),
    "train_configs": ["linear", "cosine", "ramp", "anisotropic", "pred_xstart"],
    "slice_a": ["linear", "ramp", "pred_xstart"],
    "slice_b": ["anisotropic", "cosine"],
}
if _scratch_anomaly is None:
    raise FileNotFoundError(
        "No EAF scratch project dir found under /scratch/7Day*/munjung/. "
        f"Pools seen: {_pools or ['<none>']}"
    )
out = Path("/exp/sbnd/data/users/munjung/anomaly-detection/training/eaf_discover.json")
try:
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(discover, indent=2))
    print("wrote", out)
except Exception as e:
    out = APP_ROOT / "train" / "eaf_discover.json"
    out.write_text(json.dumps(discover, indent=2))
    print("data write failed ({}); wrote".format(e), out)
print(json.dumps(discover, indent=2))


## Import training flag scripts from EAF into the app repo

Copies any `*flags*.sh` / `model_flags*.sh` found next to the diffusion checkout into
`configs/train_flags/imported/` so the training notebooks can prefer them.

Local canonical bundles for the five configs already live in `configs/train_flags/`
(`linear`, `cosine`, `ramp`, `anisotropic`, `pred_xstart`).

Two-slice training notebooks:
- `01a_TrainDiffusion_sliceA.ipynb` — linear → ramp → pred_xstart
- `01b_TrainDiffusion_sliceB.ipynb` — anisotropic → cosine


In [ ]:
IMPORT_DIR = APP_ROOT / "configs" / "train_flags" / "imported"
IMPORT_DIR.mkdir(parents=True, exist_ok=True)

search_roots = []
if DIFFUSION_ROOT:
    search_roots.append(Path(DIFFUSION_ROOT))
search_roots += [
    Path.home() / "diffusion-anomaly",
    Path("/scratch/7DayLifetime/munjung/diffusion-anomaly"),
    Path("/scratch/7DayLifetime/munjung/ICARUS-anomaly"),
    Path("/scratch/7DayLifetime/munjung/anomaly-detection"),
]

patterns = ("*flags*.sh", "model_flags*.sh", "*_flags.sh")
copied = []
seen = set()
for root in search_roots:
    if not root.exists():
        continue
    for pat in patterns:
        for src in list(root.glob(pat)) + list(root.glob("*/" + pat)):
            if not src.is_file():
                continue
            key = f"{src.parent.name}__{src.name}" if src.parent != root else src.name
            if key in seen:
                continue
            seen.add(key)
            dst = IMPORT_DIR / (key if "__" in key else src.name)
            shutil.copy2(src, dst)
            copied.append((str(src), str(dst)))

print(f"imported {len(copied)} flag scripts → {IMPORT_DIR}")
for a, b in copied[:40]:
    print(" ", a, "→", b)

for name in ("linear", "cosine", "ramp", "anisotropic", "pred_xstart"):
    p = APP_ROOT / "configs" / "train_flags" / f"{name}.sh"
    print(f"local {name}:", "OK" if p.is_file() else "MISSING", p)


In [ ]:
# Refresh discover json after flag import
discover["train_flags_local"] = str(APP_ROOT / "configs" / "train_flags")
discover["train_flags_imported"] = str(IMPORT_DIR)
discover["train_configs"] = ["linear", "cosine", "ramp", "anisotropic", "pred_xstart"]
discover["slice_a"] = ["linear", "ramp", "pred_xstart"]
discover["slice_b"] = ["anisotropic", "cosine"]
discover["diffusion_root"] = str(DIFFUSION_ROOT) if DIFFUSION_ROOT else discover.get("diffusion_root")

out = Path("/exp/sbnd/data/users/munjung/anomaly-detection/training/eaf_discover.json")
try:
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(discover, indent=2))
    print("wrote", out)
except Exception as e:
    out = APP_ROOT / "train" / "eaf_discover.json"
    out.write_text(json.dumps(discover, indent=2))
    print("fallback wrote", out, "because", e)
pprint(discover)
